# 🧠 Fine-Tune LLaMA 3.2 — Trading Brain v2 (with Ichimoku Cloud)

### Dataset: `training_data_v2_ichimoku.jsonl`
- **~495,642 examples** from **~440 tickers** (S&P 500 + Top Russell 100)
- **5 years** of daily data per ticker
- Includes **Ichimoku Cloud** indicators (trend, TK cross, cloud support/resistance)

### ⚠️ IMPORTANT - Run Cells In Order:
1. Enable **GPU T4 x2** + **Internet** in Settings
2. Run **Cell 1** → wait ~5 min → **Restart Runtime**
3. Run **Cell 2** → verify ✅ → continue Cell 3 onward
4. If session restarts mid-training, re-run Cell 1 then all cells in order

### Checkpointing:
- Saves every **500 steps** so you never lose more than ~15 min of progress
- Auto-resumes from last checkpoint if session restarts

### Changes from v1:
- Training data includes Ichimoku Cloud indicators
- Dataset 4x larger (495K vs 119K examples)
- `gradient_accumulation_steps` increased to 8 (effective batch = 16)
- `num_train_epochs` = 3 (adjusted for larger dataset)
- `warmup_steps` increased to 100 (smoother learning curve)
- `save_steps` = 1000 (less checkpoint overhead for longer runs)

## 1. Install Dependencies (Run FIRST, then restart runtime)

In [ ]:
# ─── Fix Kaggle environment conflicts ───────────────────────────────
!pip uninstall torchao -y
!pip install --upgrade --force-reinstall --no-cache-dir --no-deps unsloth unsloth_zoo
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes

print('\n' + '='*60)
print('✅ Installation complete!')
print('⚠️  Now RESTART RUNTIME, then run Cell 2.')
print('='*60)

## 2. Verify Installation (run after restart)

In [ ]:
# Safety net: uninstall torchao if it crept back
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', 'torchao', '-y'],
               capture_output=True)

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:     {torch.cuda.get_device_name(0)}')

from unsloth import FastLanguageModel
print('Unsloth: ✅ imported successfully')
print('\n✅ All good! Continue to Cell 3.')

## 3. Detect Platform & Configure Paths

**Dataset filename:** `training_data_v2_ichimoku.jsonl`

Upload this file as a Kaggle Dataset or to Google Drive before running.

In [ ]:
import os, json, shutil
from pathlib import Path

# ─── v2 Dataset filename ───────────────────────────────────────────
DATASET_FILENAME = 'training_data_v2_ichimoku.jsonl'

IS_KAGGLE = os.path.exists('/kaggle')
IS_COLAB  = 'COLAB_GPU' in os.environ or os.path.exists('/content')

if IS_KAGGLE:
    PLATFORM = 'Kaggle'
    # Search for the v2 dataset in all Kaggle input directories
    jsonl_files = list(Path('/kaggle/input').rglob(DATASET_FILENAME))
    if not jsonl_files:
        # Fallback: try any .jsonl file
        jsonl_files = list(Path('/kaggle/input').rglob('*.jsonl'))
    if jsonl_files:
        DATA_PATH = str(jsonl_files[0])
    else:
        DATA_PATH = f'/kaggle/input/training-data-v2/{DATASET_FILENAME}'
    
    CHECKPOINT_DIR = '/kaggle/working/checkpoints'
    OUTPUT_DIR     = '/kaggle/working/outputs'
    FINAL_DIR      = '/kaggle/working/trading-brain-v2'
    
elif IS_COLAB:
    PLATFORM = 'Colab'
    from google.colab import drive
    drive.mount('/content/drive')
    
    DATA_PATH      = f'/content/drive/MyDrive/{DATASET_FILENAME}'
    CHECKPOINT_DIR = '/content/drive/MyDrive/trading-brain-v2-checkpoints'
    OUTPUT_DIR     = '/content/outputs'
    FINAL_DIR      = '/content/trading-brain-v2'
else:
    raise RuntimeError('Unknown platform. Run this on Kaggle or Google Colab.')

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Platform:       {PLATFORM}')
print(f'Data:           {DATA_PATH}')
print(f'Checkpoints:    {CHECKPOINT_DIR}')
print(f'Output:         {OUTPUT_DIR}')
print(f'Final model:    {FINAL_DIR}')

# Verify data file exists
if os.path.exists(DATA_PATH):
    size_mb = os.path.getsize(DATA_PATH) / (1024 * 1024)
    print(f'\n✅ Dataset found: {size_mb:.1f} MB')
else:
    print(f'\n❌ Dataset NOT found at {DATA_PATH}')
    print('   Upload it as a Kaggle Dataset or to Google Drive first.')

## 4. Load Training Data

In [ ]:
with open(DATA_PATH, 'r') as f:
    data = [json.loads(line) for line in f]

print(f'Loaded {len(data):,} training examples')
print(f'\nSample instruction:')
print(data[0]['instruction'][:300])
print(f'\nSample output:')
print(data[0]['output'][:300])

# Verify Ichimoku data is present
ichimoku_count = sum(1 for d in data[:1000] if 'Ichimoku' in d['instruction'])
print(f'\nIchimoku data present: {ichimoku_count}/1000 first examples ({ichimoku_count/10:.0f}%)')
if ichimoku_count > 0:
    print('✅ Ichimoku Cloud indicators confirmed in training data')
else:
    print('⚠️ WARNING: No Ichimoku data found — you may have the wrong dataset file')

## 5. Check for Existing Checkpoints (Resume Support)

In [ ]:
def find_latest_checkpoint(checkpoint_dir, output_dir):
    """Find the most recent checkpoint from either directory."""
    all_checkpoints = []
    
    for search_dir in [checkpoint_dir, output_dir]:
        search_path = Path(search_dir)
        if search_path.exists():
            for d in search_path.iterdir():
                if d.is_dir() and d.name.startswith('checkpoint-'):
                    try:
                        step = int(d.name.split('-')[-1])
                        all_checkpoints.append((step, str(d)))
                    except ValueError:
                        pass
    
    if all_checkpoints:
        all_checkpoints.sort(key=lambda x: x[0])
        latest_step, latest_path = all_checkpoints[-1]
        print(f'✅ Found checkpoint: step {latest_step}')
        print(f'   Path: {latest_path}')
        print(f'   Will RESUME training from this point.')
        return latest_path
    
    print('No existing checkpoints found. Starting fresh.')
    return None

RESUME_FROM = find_latest_checkpoint(CHECKPOINT_DIR, OUTPUT_DIR)

# Show all checkpoint dirs for debugging
print(f'\n--- Checkpoint directories ---')
for d in [CHECKPOINT_DIR, OUTPUT_DIR]:
    p = Path(d)
    if p.exists():
        contents = list(p.iterdir())
        print(f'{d}: {[c.name for c in contents]}')
    else:
        print(f'{d}: does not exist')

## 6. Load Model

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Llama-3.2-3B-Instruct',
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

print(f'Model loaded: {model.num_parameters():,} parameters')

## 7. Add LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

trainable, total = model.get_nb_trainable_parameters()
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

## 8. Prepare Dataset

In [ ]:
from datasets import Dataset

def format_example(example):
    messages = [
        {'role': 'system', 'content': example['system']},
        {'role': 'user', 'content': example['instruction']},
        {'role': 'assistant', 'content': example['output']},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {'text': text}

dataset = Dataset.from_list(data)
dataset = dataset.map(format_example, num_proc=2)

print(f'Dataset: {len(dataset):,} examples')
print(f'Sample (first 400 chars):')
print(dataset[0]['text'][:400])

# Estimate training steps
BATCH_SIZE = 2
GRAD_ACCUM = 8
NUM_EPOCHS = 3
effective_batch = BATCH_SIZE * GRAD_ACCUM
steps_per_epoch = len(dataset) // effective_batch
total_steps = steps_per_epoch * NUM_EPOCHS
print(f'\n--- Training Plan ---')
print(f'Effective batch size: {effective_batch}')
print(f'Steps per epoch:     {steps_per_epoch:,}')
print(f'Total steps:         {total_steps:,}')
print(f'Est. time (T4):      ~{total_steps * 2 / 3600:.1f} hrs')

## 9. Train with Checkpointing

**v2 config changes:**
- `gradient_accumulation_steps = 8` (was 4) — effective batch 16
- `num_train_epochs = 3` — sufficient for 495K examples
- `warmup_steps = 100` (was 50) — smoother ramp for larger dataset
- `save_steps = 1000` (was 500) — less checkpoint I/O overhead

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
import logging

logging.basicConfig(level=logging.INFO)

NUM_EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM = 8   # v2: increased from 4 → 8 (effective batch = 16)

training_args = TrainingArguments(
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=100,              # v2: increased from 50
    num_train_epochs=NUM_EPOCHS,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,              # v2: reduced logging frequency
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    seed=42,
    output_dir=OUTPUT_DIR,
    report_to='none',
    save_strategy='steps',
    save_steps=1000,               # v2: 1000 (was 500) to reduce I/O
    save_total_limit=3,            # Keep only last 3 checkpoints
    average_tokens_across_devices=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=True,
    args=training_args,
)

print('=' * 60)
print(f'Training config: {NUM_EPOCHS} epochs × {len(dataset):,} examples')
print(f'Batch: {BATCH_SIZE} × {GRAD_ACCUM} grad_accum = {BATCH_SIZE*GRAD_ACCUM} effective')
if RESUME_FROM:
    print(f'RESUMING from checkpoint: {RESUME_FROM}')
    stats = trainer.train(resume_from_checkpoint=RESUME_FROM)
else:
    print('Starting training from scratch...')
    stats = trainer.train()

print(f'\n{"=" * 60}')
print(f'Training complete!')
print(f'Steps: {stats.global_step} | Loss: {stats.training_loss:.4f}')

## 10. Copy Checkpoints to Persistent Storage

In [ ]:
import glob

src_checkpoints = sorted(glob.glob(f'{OUTPUT_DIR}/checkpoint-*'))

for src in src_checkpoints:
    name = os.path.basename(src)
    dest = os.path.join(CHECKPOINT_DIR, name)
    if not os.path.exists(dest):
        print(f'Saving {name} → {CHECKPOINT_DIR}/')
        shutil.copytree(src, dest)
    else:
        print(f'{name} already saved')

print(f'\n✅ Checkpoints saved to: {CHECKPOINT_DIR}')
print('   If the notebook restarts, re-run from the top to resume.')

## 11. Test the Fine-Tuned Model

Test with an Ichimoku-enriched prompt to verify the model learned the new indicators.

In [ ]:
FastLanguageModel.for_inference(model)

# v2 test prompt: includes Ichimoku Cloud data
test_prompt = (
    'Analyze AAPL on 2024-03-15. | Price: $172.50 | Trend: uptrend | '
    'RSI: 68.5 (moderately_bullish) | MACD: 1.2500 | MACD Signal: 0.8900 | '
    'SMA20: $170.30 | SMA50: $165.20 | Volume: 1.35x average | '
    'BB Width: 0.0450 | BB Position: 0.82 | '
    'Ichimoku: bullish, TK cross: bullish | Cloud: $160.50-$163.80'
)

messages = [
    {'role': 'system', 'content': 'You are an expert stock market analyst trained on historical pattern data.'},
    {'role': 'user', 'content': test_prompt},
]

inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
).to('cuda')

outputs = model.generate(input_ids=inputs, max_new_tokens=500, temperature=0.3)
response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

print('Test Response (should include Ichimoku in reasoning):')
print(response)

# Parse and validate
try:
    result = json.loads(response)
    print(f'\n--- Parsed ---')
    print(f'Recommendation: {result.get("recommendation")}')
    print(f'Confidence:     {result.get("confidence")}')
    print(f'Target:         {result.get("target_price")}')
    print(f'Stop Loss:      {result.get("stop_loss")}')
    if 'ichimoku' in result.get('reasoning', '').lower() or 'cloud' in result.get('reasoning', '').lower():
        print('\n✅ Model references Ichimoku Cloud in reasoning!')
    else:
        print('\n⚠️ Model did not mention Ichimoku — may need more training')
except json.JSONDecodeError:
    print('\n⚠️ Response is not valid JSON — model may need more training')

## 12. Export as GGUF for Ollama

In [ ]:
model.save_pretrained_gguf(
    FINAL_DIR,
    tokenizer,
    quantization_method='q4_k_m',
)

# Unsloth saves GGUF to FINAL_DIR or FINAL_DIR_gguf — check both
GGUF_DIR = None
gguf_file = None

for candidate_dir in [FINAL_DIR, f'{FINAL_DIR}_gguf']:
    if os.path.exists(candidate_dir):
        found = [f for f in os.listdir(candidate_dir) if f.endswith('.gguf')]
        if found:
            GGUF_DIR = candidate_dir
            gguf_file = found[0]
            break

if gguf_file:
    gguf_path = f'{GGUF_DIR}/{gguf_file}'
    gguf_size = os.path.getsize(gguf_path) / (1024**3)
    print(f'✅ GGUF exported: {gguf_file} ({gguf_size:.2f} GB)')
    print(f'   Location: {gguf_path}')
else:
    print('ERROR: No GGUF file found in any expected directory')
    for d in [FINAL_DIR, f'{FINAL_DIR}_gguf']:
        if os.path.exists(d):
            print(f'  {d}: {os.listdir(d)}')

## 13. Download & Deploy Instructions

In [ ]:
if not gguf_file:
    print('ERROR: No GGUF file available. Re-run Cell 12 first.')
else:
    if IS_COLAB:
        drive_dest = f'/content/drive/MyDrive/{gguf_file}'
        shutil.copy(f'{GGUF_DIR}/{gguf_file}', drive_dest)
        print(f'Saved to Google Drive: {drive_dest}')
    
    print()
    print('=' * 60)
    print('  NEXT STEPS — Deploy as trading-brain-v2')
    print('=' * 60)
    
    if IS_KAGGLE:
        print(f'''
1. Click "Save Version" (top right) → Save & Run All
2. Go to Output tab → download {gguf_file}
''')
    
    print(f'''3. Place the .gguf file in a folder on your PC
4. Create a file called "Modelfile" next to it:

   FROM ./{gguf_file}
   PARAMETER temperature 0.3
   PARAMETER num_predict 2048
   SYSTEM "You are an expert stock market analyst. Given technical indicators for a stock, predict the most likely outcome and provide your analysis as a JSON object with recommendation, confidence, target_price, stop_loss, time_horizon, risk_level, and reasoning."

5. Run:  ollama create trading-brain-v2 -f Modelfile
6. Test: ollama run trading-brain-v2 "Analyze AAPL. Price: $195. RSI: 55. Ichimoku: bullish, TK cross: bullish. Cloud: $180-$188."
7. Edit .env:  OLLAMA_MODEL=trading-brain-v2
8. Restart the backend server!
''')